### P2.2 — ML Foundations | Python to Gen AI Course
### P2.2.5 – Unsupervised Learning: Clustering — Customer Segmentation

---
## 🔁 Quick Recap — The 3-Step Framework

Before picking an algorithm, always ask 3 questions:

```
Step 1 → What TYPE?       Supervised / Unsupervised / Reinforcement
Step 2 → What TASK?       Regression (predict a number) / Classification (predict a category) / Clustering (find groups)
Step 3 → What ALGORITHM?  The task tells you which algorithm to pick
```

In this module, we cover **one Unsupervised Learning algorithm**:

| Step | This Notebook |
|---|---|
| Type | **Unsupervised** — no labels, machine finds structure in raw data |
| Task | **Clustering** — discover natural groups hidden in the data |
| Algorithm | **K-Means** — assigns every data point to one of K groups ← **we start here** |

Supervised algorithms were covered in:
- **P2.2.3** — Linear Regression (predict a continuous value: house prices)
- **P2.2.4** — Naive Bayes (classify into categories: spam vs ham)

> **We won't build K-Means from scratch.** We use `scikit-learn` — the same library professionals use.  
> Your job is to understand *when* to use it, *what it does*, and *how to interpret results*.

---
## 🌍 Real Life — Where Clustering Is Used Every Day

**🛒 Amazon / Flipkart — Customer Segments**

Amazon doesn't manually label every customer as "electronics buyer" or "budget shopper."  
They run clustering on purchase history — the machine finds natural groups.  
Then each group gets different deals, recommendations, and emails.

**Other real-world examples:**
- 🎵 **Spotify** — groups users by listening behaviour → creates "Daily Mix" playlists
- 🍕 **Zomato / Swiggy** — segments users by cuisine preference and order time
- 🏥 **Healthcare** — clusters patients with similar symptoms for research
- 📰 **News apps** — groups articles by topic automatically (no one labels each article)

> **Any time a company says "people like you also..." → that's clustering at work.**  
> No human labeled those groups. The machine found them.

---

### 🧭 Which Clustering Algorithm Should I Use?

| Situation | Best Algorithm | Why |
|---|---|---|
| You know how many groups you want | **K-Means** ← *this notebook* | Fast, scalable, works well on spherical clusters |
| You don't know K and want a hierarchy | **Hierarchical Clustering** | Builds a tree — you cut at any level to get K groups |
| Clusters are irregular / oddly shaped | **DBSCAN** | Finds dense regions; handles noise and outliers |
| Very high-dimensional data (text, images) | **K-Means + dimensionality reduction** | Reduce dimensions first (PCA), then cluster |
| You want soft / probabilistic membership | **Gaussian Mixture Models** | Each point gets a probability of belonging to each group |

> **For this course:** K-Means is the right starting point.  
> It covers 80% of real-world clustering use cases and is the conceptual foundation for embeddings in Gen AI.

---
## 💻 Let's Build It — Customer Segmentation

We have 8 customers, each described by their purchase category and city.  
**No labels** — we don't know in advance which group they belong to.

```
Customer                  (no label)
───────────────────────────────────
"Electronics Mumbai"         ?
"Groceries Delhi"            ?
"Clothing Mumbai"            ?
"Electronics Delhi"          ?
"Groceries Mumbai"           ?
"Clothing Delhi"             ?
"Electronics Bangalore"      ?
"Groceries Bangalore"        ?
```

We'll vectorize the text and let **K-Means (K=3)** find the natural groups.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

# ── DATA (no labels — unsupervised) ──────────────────────────────
customers = [
    "Electronics Mumbai",
    "Groceries Delhi",
    "Clothing Mumbai",
    "Electronics Delhi",
    "Groceries Mumbai",
    "Clothing Delhi",
    "Electronics Bangalore",
    "Groceries Bangalore",
]

# customers = [
#     "Electronics",
#     "Groceries",
#     "Clothing",
#     "Electronics",
#     "Groceries",
#     "Clothing",
#     "Electronics",
#     "Groceries",
# ]  

"""
Vocabulary = [bangalore, clothing, delhi, electronics, groceries, mumbai]

"Electronics Mumbai"    →  [0, 0, 0, 1, 0, 1]
"Groceries Delhi"       →  [0, 0, 1, 0, 1, 0]
"Clothing Mumbai"       →  [0, 1, 0, 0, 0, 1]
"Electronics Delhi"     →  [0, 0, 1, 1, 0, 0]
"Groceries Mumbai"      →  [0, 0, 0, 0, 1, 1]
"Clothing Delhi"        →  [0, 0, 1, 1, 0, 0]  
"Electronics Bangalore" →  [1, 0, 0, 1, 0, 0]
"Groceries Bangalore"   →  [1, 0, 0, 0, 1, 0]
"""

# ── VECTORIZATION (text → numbers) ───────────────────────────────
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(customers).toarray()

print("=== VOCABULARY USED ===")
print(vectorizer.get_feature_names_out())


# ── PHASE 1 : TRAIN ───────────────────────────────────────────────
# No split needed — unsupervised uses all data to find structure
# No evaluate phase — no true labels exist to compare against
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X)


# ── PHASE 2 : INFERENCE ───────────────────────────────────────────
# Assign each customer to a cluster
cluster_labels = kmeans.labels_

print("\n=== CUSTOMER SEGMENTS ===")
print(f"{'Customer':<30} {'Cluster':>8}")
print("-" * 40)
for customer, cluster in zip(customers, cluster_labels):
    print(f"{customer:<30} {'Group ' + str(cluster):>8}")


# ── PHASE 3 : INTERPRET ───────────────────────────────────────────
# No right or wrong answer — we read the groups and make sense of them
print("\n=== WHO IS IN EACH GROUP? ===")
for group_id in sorted(set(cluster_labels)):
    members = [c for c, g in zip(customers, cluster_labels) if g == group_id]
    print(f"\nGroup {group_id}: {members}")

=== VOCABULARY USED ===
['bangalore' 'clothing' 'delhi' 'electronics' 'groceries' 'mumbai']

=== CUSTOMER SEGMENTS ===
Customer                        Cluster
----------------------------------------
Electronics Mumbai              Group 1
Groceries Delhi                 Group 2
Clothing Mumbai                 Group 0
Electronics Delhi               Group 1
Groceries Mumbai                Group 0
Clothing Delhi                  Group 2
Electronics Bangalore           Group 1
Groceries Bangalore             Group 0

=== WHO IS IN EACH GROUP? ===

Group 0: ['Clothing Mumbai', 'Groceries Mumbai', 'Groceries Bangalore']

Group 1: ['Electronics Mumbai', 'Electronics Delhi', 'Electronics Bangalore']

Group 2: ['Groceries Delhi', 'Clothing Delhi']


Group 1 →  Electronics Mumbai, Electronics Delhi, Electronics Bangalore
           Common vector signal : electronics=1 across all three
           ✅ This one worked — electronics dominated over city

Group 2 →  Groceries Delhi, Clothing Delhi
           Common vector signal : delhi=1 across both
           ❌ City won over shopping type here

Group 0 →  Clothing Mumbai, Groceries Mumbai, Groceries Bangalore
           Common vector signal : mumbai=1 or groceries=1 — mixed signal
           ❌ Neither city nor category cleanly won

---
```
clustering by shopping type only 

customers = [
    "Electronics",
    "Groceries",
    "Clothing",
    "Electronics",
    "Groceries",
    "Clothing",
    "Electronics",
    "Groceries",
]  
```

---
## ✅ Pros & Limitations of K-Means

**Pros:**
- Works without any labels — perfect when you have data but no pre-defined categories
- Simple and fast — scales to millions of customers
- Useful starting point for understanding your data before building supervised models

**Limitations:**
- You must **choose K** in advance — how many groups? Not always obvious
- Sensitive to the initial random placement of centers — can give different results each run
- Assumes roughly round/equal-sized clusters — struggles with unusual shapes or very unequal groups

---

> For K-Means, the practical check is simple: **do the groups make real-world sense?**  
> If "Tech Buyers" and "Daily Shoppers" are clearly different segments — your K is about right.

---
## ✅ Summary — What You Learned

| Concept | Key Point |
|---|---|
| Unsupervised Learning | No labels — machine finds structure in raw data |
| K-Means Clustering | Assigns data points to K groups based on similarity |
| No train-test split | We're not predicting a right answer — we're discovering groups |
| Interpreting clusters | You assign meaning to groups *after* the model finds them |
| GenAI link | Embeddings = clustering of meaning in high-dimensional space; powers semantic search & RAG |

---

## 🎉 Module P2.2 — ML Foundations Complete!

You've covered the 5 essential ML concepts needed to understand Gen AI:

| Notebook | ML Type | Algorithm | Real Example |
|---|---|---|---|
| P2.2.1 Introduction | Foundations | What is ML? | y = 2x pattern discovery |
| P2.2.2 Decision Framework | Strategy | How to pick an algorithm | 3-step framework + ML workflow |
| P2.2.3 Linear Regression | Supervised | Linear Regression | House price prediction |
| P2.2.4 Classification | Supervised | Naive Bayes | Spam email detection |
| P2.2.5 Clustering | Unsupervised | K-Means | Customer segmentation |

**The bridge is now built:**
```
Rule-Based AI  →  Machine Learning  →  Deep Learning  →  Gen AI
  (P2.1)            (P2.2.1–2.2.5)        (Next)         (End Goal)
```

**👉 Next Module: Deep Learning & Neural Networks — the final step before Gen AI**